<a href="https://colab.research.google.com/github/Jhon-Victor-Ramos/pibic-deteccao-anomalias/blob/main/Implementa%C3%A7%C3%A3o_do_Isolation_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import random as rd

## Criando a *Isolation Tree* (*iTree*)

In [11]:
# Criando o nó (basicamente o mesmo nó de uma árvore binária)
# Como ele é uma "coisa" no código, então ele que é a classe mesmo
class Node:
    def __init__(self, left = None, right = None, split_value = None, size = None):
        self.left = left
        self.right = right
        self.split_value = split_value
        self.size = size

In [12]:
# Criando a árvore de isolamento (iTree)
def build_iTree(X, current_depth, max_depth):
    # Caso base: se a quantidade de amostras for menor que ou igual 1 OU a profundidade atual foi maior que ou igual a profundidade máxima
    n_samples = X.shape[0]
    if n_samples <= 1 or current_depth >= max_depth:
        return Node(size = n_samples)

    # Pegando o menor e o maior valor do array para conseguir sortear um número dentro desse intervalo e realizar o corte
    min_val = X.min()
    max_val = X.max()

    if min_val == max_val:
        return Node(size = n_samples)

    # Fazendo o sorteio para criar a divisão
    split_value = rd.uniform(min_val, max_val)

    # Aplicando máscara booleana para conseguir fazer um filtro dos valores
    left_mask = X < split_value
    right_mask = X >= split_value

    # Usando a máscara para conseguir "transferir" os números para seus grupos corretos
    left_group = X[left_mask]
    right_group = X[right_mask]

    # Aplicando a recursão
    left_child = build_iTree(left_group, current_depth+1, max_depth)
    right_child = build_iTree(right_group, current_depth+1, max_depth)

    return Node(left = left_child, right = right_child, split_value =split_value)

In [13]:
# Criando a função para buscar o tanto que o nó passou por caminhos lógicos
def path_length(x, node, current_depth=0):
    # Caso base: chegou no nó folha (ambos os lados são vazios)
    if node.left is None and node.right is None:
        return current_depth

    if x < node.split_value:
        return path_length(x, node.left, current_depth + 1)
    else:
        return path_length(x, node.right, current_depth + 1)

## Criando a Isolation Forest (*iForest*)

In [14]:
def build_forest(X, n_trees, max_depth):
    forest = []

    while (n_trees > 0):
        # TODO: Garantir que o código não faça sempre 256 amostras. Pois se o X for menor que isso, ele acessará valores inexistentes
        new_X = np.random.choice(X, size = 256, replace = False, p = None)
        new_tree = build_iTree(new_X, 0, max_depth)
        forest.append(new_tree)
        n_trees -= 1

    return forest


In [15]:
def forest_path_length(x, forest):
    paths = []

    for f in forest:
        forest_path = path_length(x, f)
        paths.append(forest_path)

    paths_mean = sum(paths) / len(paths)

    return paths_mean